In [1]:
import re
import pandas as pd
from pathlib import Path
from datetime import datetime

import SCRIPTS.jsonDownloader as jd
from SCRIPTS.redditLinkRetriever import fetch_saved_post_links, save_links_txt
from SCRIPTS.mediaDownloader import download_embedded_media
from SCRIPTS.mediaOrganizer import organize_downloads
from SCRIPTS.redgifDownloader import process_external
from SCRIPTS.cloudflareUploader import upload_media
from SCRIPTS.r2_audit import audit_local_vs_r2

In [2]:
# https://old.reddit.com/prefs/apps
# Set to True to avoid making any changes
DRY_RUN_MEDIA = False
DRY_RUN_ORGANIZE = False

DRY_RUN_CLOUDFLARE = False   # True = preview only, no upload
ACCOUNT_INDEX = 2            # choose which R2 account (1, 2, ...)
CHECK_ONLY = False           # True = just check existence, no upload

DRY_RUN_FINAL = False

In [3]:
# Retrieve saved post links for the specified user
links = fetch_saved_post_links()

In [4]:
len(links)

198

In [5]:
links[:5]

['https://www.reddit.com/r/bangmybully/comments/1oy583g/shes_completely_comfortable_being_his_slut_but',
 'https://www.reddit.com/r/ABGHeavens/comments/1oyb28z/perfect_and_petite_chinese_abg',
 'https://www.reddit.com/r/IWantToBeHerHentai2/comments/1oynbzu/is_it_too_much_to_ask_to_be_her_one_day',
 'https://www.reddit.com/r/ABGHeavens/comments/1oyfyvl/anyone_know_the_scene',
 'https://www.reddit.com/r/bangmybully/comments/1oxhrat/next_time_dont_send_your_mom_to_fight_your']

# NEW POST VALIDATION

This section validates new posts from reddits saved folder

In [6]:
csv_path = Path("ordered_posts.csv")
raw_df = pd.read_csv(csv_path)

POST_ID_RE = re.compile(r"/comments/([a-z0-9]+)(?:[/?#]|$)", re.IGNORECASE)
SHORT_RE   = re.compile(r"redd\.it/([a-z0-9]+)(?:[/?#]|$)", re.IGNORECASE)
max_order_num = raw_df.order_num.max()

def strip_trailing_slash(url: str) -> str:
    # remove trailing slashes only at the very end (doesn't touch scheme)
    return url.rstrip("/")

def extract_post_id(url: str) -> str | None:
    """
    Try to extract a post id from:
      - standard permalink: .../comments/<postid>/...
      - shortlink: https://redd.it/<postid>
    """
    m = POST_ID_RE.search(url)
    if m:
        return m.group(1)
    m = SHORT_RE.search(url)
    if m:
        return m.group(1)
    return None

existing_ids = set(str(x).lower() for x in raw_df.get("post_id", pd.Series([])).dropna())

new_rows = []
next_order = max_order_num + 1
seen_in_batch = set()  # avoid duplicates within this run

for raw_link in reversed(links):
    link = strip_trailing_slash(raw_link)
    post_id = extract_post_id(link)
    if not post_id:
        continue
    pid = post_id.lower()

    # Only add if NOT already in CSV and not already queued this batch
    if pid in existing_ids or pid in seen_in_batch:
        continue

    new_rows.append({
        "order_num": next_order,
        "link": link,
        "post_id": post_id,
        "date_added": datetime.utcnow().isoformat(timespec="seconds"),
    })
    seen_in_batch.add(pid)
    next_order += 1

# Preview as a DataFrame
new_df = pd.DataFrame(new_rows)
new_df


C:\Users\minds\AppData\Local\Temp\ipykernel_6580\787629229.py:47: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  "date_added": datetime.utcnow().isoformat(timespec="seconds"),


,order_num,link,post_id,date_added
0,1400,https://www.reddit.com/r/ABGHeavens/comments/1...,1ou1peh,2025-11-16T21:24:17
1,1401,https://www.reddit.com/r/ABGHeavens/comments/1...,1ovuj7t,2025-11-16T21:24:17
2,1402,https://www.reddit.com/r/cumsluts/comments/1ow...,1ow134r,2025-11-16T21:24:17
3,1403,https://www.reddit.com/r/ABGHeavens/comments/1...,1ows3lm,2025-11-16T21:24:17
4,1404,https://www.reddit.com/r/IWantToBeHerHentai2/c...,1oxkbge,2025-11-16T21:24:17
5,1405,https://www.reddit.com/r/ffmFantasy/comments/1...,1ox1tt3,2025-11-16T21:24:17
6,1406,https://www.reddit.com/r/ABGHeavens/comments/1...,1owj55f,2025-11-16T21:24:17
7,1407,https://www.reddit.com/r/ABGHeavens/comments/1...,1owbd5v,2025-11-16T21:24:17
8,1408,https://www.reddit.com/r/bangmybully/comments/...,1owuay6,2025-11-16T21:24:17
9,1409,https://www.reddit.com/r/cumsluts/comments/1ox...,1ox4x9i,2025-11-16T21:24:17


In [7]:
final_df = pd.concat([raw_df, new_df], ignore_index=True)
final_df = final_df.sort_values(by="order_num", ascending=False).reset_index(drop=True)

In [8]:
import importlib
importlib.reload(jd)

jd.configure(
    DATA_ROOT="Out",
    SKIP_EXISTING=False,
    REPORTS_DIR="__reports",
    write_csv_to=None
    )

summary = jd.process_all(new_df["link"].tolist(), show_progress=True)
summary

  0%|          | 0/16 [00:00<?, ?post/s]

Done. Success: 16, Skipped: 0, Failed: 0


{'success': 16, 'skipped': 0, 'failed': 0}

# MEDIA DOWNLOADER
Reviews the external and media json folders in **Out/**, downloading:
- Images
- Gifs
- Videos

In [9]:
folders = ["external", "media"]
download_stats = []

# point to your inputs/outputs explicitly
for mediaType in folders:
    download_stats.append(download_embedded_media(
        media_json_dir=Path("Out/" + mediaType),   # where your *.json live
        media_out_dir=Path("Media/media_files"),  # where downloads should go
        write_fail_csv_to=Path("__reports/media_report" + datetime.now().strftime("%Y%m%d-%H%M%S") + ".csv"),
        show_progress=True,
    ))

download_stats

[{'downloaded': 0,
  'failed': 8,
  'skipped': 0,
  'fail_rows': [{'id': '1ow134r', 'reason': 'no_reddit_media_url'},
   {'id': '1owuay6', 'reason': 'no_reddit_media_url'},
   {'id': '1owwlxg', 'reason': 'no_reddit_media_url'},
   {'id': '1ox1tt3', 'reason': 'no_reddit_media_url'},
   {'id': '1ox4x9i', 'reason': 'no_reddit_media_url'},
   {'id': '1oxhrat', 'reason': 'no_reddit_media_url'},
   {'id': '1oxkbge', 'reason': 'no_reddit_media_url'},
   {'id': '1oy583g', 'reason': 'no_reddit_media_url'}],
  'out_dir': WindowsPath('Media/media_files'),
  'json_dir': WindowsPath('Out/external')},
 {'downloaded': 39,
  'failed': 0,
  'skipped': 0,
  'fail_rows': [],
  'out_dir': WindowsPath('Media/media_files'),
  'json_dir': WindowsPath('Out/media')}]

In [10]:
move_stats = organize_downloads(
    input_dir="Media/media_files",  # where your downloader wrote files
    output_dir="Media",             # where Images/, Videos/, Gifs/ live
    strategy="move",
    conflict="move_existing",
    show_progress=True,
    dry_run=DRY_RUN_ORGANIZE,       # set True to preview
    prune_empty_galleries=True,     # remove empty src folders after moving
)

move_stats

Organizing media:   0%|          | 0/39 [00:00<?, ?file/s]

{'moved': 39,
 'copied': 0,
 'linked': 0,
 'skipped': 0,
 'unknown': 0,
 'dry_run': False,
 'strategy': 'move',
 'conflict': 'move_existing',
 'input_dir': 'S:\\minds\\Desktop\\Downloader and Reddit System\\Saved-Reddit\\Media\\media_files',
 'output_dir': 'S:\\minds\\Desktop\\Downloader and Reddit System\\Saved-Reddit\\Media',
 'errors': [],
 'created_dirs': {'S:\\minds\\Desktop\\Downloader and Reddit System\\Saved-Reddit\\Media\\Gifs',
  'S:\\minds\\Desktop\\Downloader and Reddit System\\Saved-Reddit\\Media\\Gifs\\1ovuj7t',
  'S:\\minds\\Desktop\\Downloader and Reddit System\\Saved-Reddit\\Media\\Gifs\\1ows3lm',
  'S:\\minds\\Desktop\\Downloader and Reddit System\\Saved-Reddit\\Media\\Images',
  'S:\\minds\\Desktop\\Downloader and Reddit System\\Saved-Reddit\\Media\\Images\\1ovuj7t',
  'S:\\minds\\Desktop\\Downloader and Reddit System\\Saved-Reddit\\Media\\Images\\1owbd5v',
  'S:\\minds\\Desktop\\Downloader and Reddit System\\Saved-Reddit\\Media\\Images\\1ows3lm',
  'S:\\minds\\Deskt

# REDGIF DOWNLOADER

Downloads redgifs from external json folder in **Out/**

In [11]:
stats = process_external(
    media_json_dir=Path("Out/external"),
    media_out_dir=Path("Media/RedGiphys"),
    write_fail_csv_to=Path("__reports/redgif_report_" + datetime.now().strftime("%Y%m%d-%H%M%S") + ".csv"),
    write_links_csv_to=Path("__reports/external_links" + datetime.now().strftime("%Y%m%d-%H%M%S") + ".csv"),
    show_progress=True,
    dry_run=DRY_RUN_MEDIA,
    overwrite_downloads=False,
)

stats

Found 8 external post JSONs in Out\external


Scanning external posts:   0%|          | 0/8 [00:00<?, ?post/s]

[REDGIFS] id=1ow134r -> 1ow134r.mp4
[REDGIFS] id=1owuay6 -> 1owuay6.mp4
[REDGIFS] id=1owwlxg -> 1owwlxg.mp4
[REDGIFS] id=1ox1tt3 -> 1ox1tt3.mp4
[REDGIFS] id=1ox4x9i -> 1ox4x9i.mp4
[REDGIFS] id=1oxhrat -> 1oxhrat.mp4
[REDGIFS] id=1oxkbge -> 1oxkbge.mp4
[REDGIFS] id=1oy583g -> 1oy583g.mp4
Saved external links to: S:\minds\Desktop\Downloader and Reddit System\Saved-Reddit\__reports\external_links20251116-132552.csv


{'external_rows': [{'id': '1ow134r',
   'link': 'https://www.redgifs.com/watch/hiddendarkslategraywaterbug',
   'domain': 'www.redgifs.com'},
  {'id': '1owuay6',
   'link': 'https://www.redgifs.com/watch/braveglisteningarctichare',
   'domain': 'www.redgifs.com'},
  {'id': '1owwlxg',
   'link': 'https://www.redgifs.com/watch/suburbanecstatickusimanse',
   'domain': 'www.redgifs.com'},
  {'id': '1ox1tt3',
   'link': 'https://www.redgifs.com/watch/jollypreviousaustraliancurlew',
   'domain': 'www.redgifs.com'},
  {'id': '1ox4x9i',
   'link': 'https://www.redgifs.com/watch/woodenkeyfiddlercrab',
   'domain': 'www.redgifs.com'},
  {'id': '1oxhrat',
   'link': 'https://www.redgifs.com/watch/unwelcomeashamedblesbok',
   'domain': 'www.redgifs.com'},
  {'id': '1oxkbge',
   'link': 'https://www.redgifs.com/watch/everlastinginfinitecentipede',
   'domain': 'www.redgifs.com'},
  {'id': '1oy583g',
   'link': 'https://www.redgifs.com/watch/presentconventionalbarbet',
   'domain': 'www.redgifs.com'

# CLOUDFLARE VERIFICATION & UPLOAD

In [12]:
raw_output = []
uploadsData = []

for mediaType in ["Images", "Videos", "Gifs", "RedGiphys"]:
    try:
        result = upload_media(
            input_path=Path("Media") / mediaType,  # where local files/galleries live
            r2_prefix=mediaType,                   # must match bucket prefix
            account_idx=ACCOUNT_INDEX - 1,         # <— choose which R2 credentials to use
            dry_run=DRY_RUN_CLOUDFLARE,            # preview vs. real upload
            overwrite=False,                       # don't overwrite existing objects
            check_only=CHECK_ONLY,                 # <— enable to just check existence
        )

        raw_output.append(result)
        # choose which list you want to visualize depending on mode
        if CHECK_ONLY:
            uploadsData.extend(result["exists"] + result["missing"])
        else:
            uploadsData.extend(result["planned"])

    except Exception as e:
        print(f"⚠️ Error on {mediaType}: {e}")
        continue

⚠️ Error on Videos: Input path not found or not a directory: S:\minds\Desktop\Downloader and Reddit System\Saved-Reddit\Media\Videos


In [13]:
results_df = pd.DataFrame(uploadsData)
pd.set_option('display.max_rows', None)
results_df

,local,r2_key,bytes,content_type,status,account_index,bucket
0,S:\minds\Desktop\Downloader and Reddit System\...,Images/1ou1peh.jpeg,934105,image/jpeg,planned,1,media-archive
1,S:\minds\Desktop\Downloader and Reddit System\...,Images/1owj55f.jpeg,175426,image/jpeg,planned,1,media-archive
2,S:\minds\Desktop\Downloader and Reddit System\...,Images/1ovuj7t/01.jpg,77581,image/jpeg,planned,1,media-archive
3,S:\minds\Desktop\Downloader and Reddit System\...,Images/1ovuj7t/02.jpg,63680,image/jpeg,planned,1,media-archive
4,S:\minds\Desktop\Downloader and Reddit System\...,Images/1ovuj7t/03.jpg,31420,image/jpeg,planned,1,media-archive
5,S:\minds\Desktop\Downloader and Reddit System\...,Images/1ovuj7t/04.jpg,274068,image/jpeg,planned,1,media-archive
6,S:\minds\Desktop\Downloader and Reddit System\...,Images/1ovuj7t/06.jpg,159573,image/jpeg,planned,1,media-archive
7,S:\minds\Desktop\Downloader and Reddit System\...,Images/1ovuj7t/07.jpg,318842,image/jpeg,planned,1,media-archive
8,S:\minds\Desktop\Downloader and Reddit System\...,Images/1ovuj7t/08.jpg,80769,image/jpeg,planned,1,media-archive
9,S:\minds\Desktop\Downloader and Reddit System\...,Images/1owbd5v/01.jpg,158167,image/jpeg,planned,1,media-archive


# VERIFY UPLOAD

In [14]:
cats = ["Images", "RedGiphys", "Gifs", "Videos"]
all_rows = []

for cat in cats:
    try:
        res = audit_local_vs_r2(
            local_root=Path("Media") / cat,  # e.g., Media/Images
            r2_prefixes=[cat],               # matches your bucket key prefix
            account_indices=None,            # all accounts
            write_csv_to=None,               # (optional) per-cat CSV
            show_progress=True,
        )
        rows = res["rows"]
        for r in rows:
            r["category"] = cat
        all_rows.extend(rows)
    except Exception as e:
        print(f"⚠️ Error auditing {cat}: {e}")
        continue

audit_results = pd.DataFrame(all_rows)
pd.set_option("display.max_rows", None)
audit_results

Auditing: 100%|██████████| 13/13 [00:00<00:00, 6833.68file/s]

⚠️ Error auditing Videos: Local images root not found or not a directory: S:\minds\Desktop\Downloader and Reddit System\Saved-Reddit\Media\Videos


,local_rel,local_ext,all_expected_keys,matched,match_type,matched_prefix,matched_key,remote_ext,same_ext,matched_account_index,matched_bucket,note,category
0,1ou1peh.jpeg,.jpeg,Images/1ou1peh.jpeg,True,exact,Images,Images/1ou1peh.jpeg,.jpeg,True,1,media-archive,exact_match,Images
1,1owj55f.jpeg,.jpeg,Images/1owj55f.jpeg,True,exact,Images,Images/1owj55f.jpeg,.jpeg,True,1,media-archive,exact_match,Images
2,1ovuj7t/01.jpg,.jpg,Images/1ovuj7t/01.jpg,True,exact,Images,Images/1ovuj7t/01.jpg,.jpg,True,1,media-archive,exact_match,Images
3,1ovuj7t/02.jpg,.jpg,Images/1ovuj7t/02.jpg,True,exact,Images,Images/1ovuj7t/02.jpg,.jpg,True,1,media-archive,exact_match,Images
4,1ovuj7t/03.jpg,.jpg,Images/1ovuj7t/03.jpg,True,exact,Images,Images/1ovuj7t/03.jpg,.jpg,True,1,media-archive,exact_match,Images
5,1ovuj7t/04.jpg,.jpg,Images/1ovuj7t/04.jpg,True,exact,Images,Images/1ovuj7t/04.jpg,.jpg,True,1,media-archive,exact_match,Images
6,1ovuj7t/06.jpg,.jpg,Images/1ovuj7t/06.jpg,True,exact,Images,Images/1ovuj7t/06.jpg,.jpg,True,1,media-archive,exact_match,Images
7,1ovuj7t/07.jpg,.jpg,Images/1ovuj7t/07.jpg,True,exact,Images,Images/1ovuj7t/07.jpg,.jpg,True,1,media-archive,exact_match,Images
8,1ovuj7t/08.jpg,.jpg,Images/1ovuj7t/08.jpg,True,exact,Images,Images/1ovuj7t/08.jpg,.jpg,True,1,media-archive,exact_match,Images
9,1owbd5v/01.jpg,.jpg,Images/1owbd5v/01.jpg,True,exact,Images,Images/1owbd5v/01.jpg,.jpg,True,1,media-archive,exact_match,Images


In [15]:
if DRY_RUN_FINAL or (False in audit_results["matched"].value_counts().keys()):
    print(final_df.head(20))
    print("Dry run enabled; ordered_posts not changed")
else:
    print("Updating ordered_posts.csv with new posts...")
    final_df.to_csv("ordered_posts.csv", index=False)

Updating ordered_posts.csv with new posts...
